# Probabilistic cell typing

This notebooks guides you through the integration of ISS data with pre-existing clustered and annotated scRNAseq datasets, using Probabilistic Cell Typing (PCIseq).

Please have a look at:
https://www.nature.com/articles/s41592-019-0631-4
https://github.com/acycliq/pciSeq


Using this method, and if the genes measured by ISS have been accurately chosen for the task, it is possible to link these 2 modalities, and put on a geographical map the clusters inferred by scRNAseq in a coupled dataset.

## Import the necessary modules

In [ ]:
import ISS_postprocessing.pciseq as PCIseq

# --- Core imports ---
from pathlib import Path
import numpy as np
import pandas as pd

# --- Image I/O and processing ---
from tifffile import imread, imwrite
from scipy.sparse import load_npz, coo_matrix
from skimage.segmentation import mark_boundaries, find_boundaries
from skimage.morphology import disk
from scipy.ndimage import binary_dilation

# --- Plotting ---
import matplotlib.pyplot as plt
import seaborn as sns


## Read the single cell RNA sequencing dataset

In this step we can either input a clustered scRNAseq object, or even just a table with the average expression data per cluster.

In [ ]:
sc_file = '/path/to/sc_file.csv/'

In [ ]:
sc_file = ('/mnt/DATA/ext_home/saga/mean_expression_sc_gallus_input_for_pciseq_82genes.csv')

In [ ]:
scRNAseq = pd.read_csv(sc_file, header=None, index_col=0, compression=None, dtype=object)
scRNAseq = scRNAseq.rename(columns=scRNAseq.iloc[0], copy=False).iloc[1:]
scRNAseq = scRNAseq.astype(float)


In [ ]:
# We can show the data to have a look and confirm everything looks as it should.
scRNAseq

### Import the segmentation mask and the ISS data

In the following steps, we first load the two main inputs required for building the ISS–scRNAseq integration dataset:

* **Segmentation mask** (`coo_file`):
  A sparse label image (saved in `.npz` format) where each pixel is assigned to a cell ID. This defines the spatial boundaries of individual cells.

* **Decoded ISS spots** (`spots_file`):
  A table (CSV or similar) containing the coordinates and gene identities of all decoded spots. Each spot will later be assigned to a cell in the segmentation mask.

Together, these inputs are combined to construct a spatially resolved **gene-by-cell count matrix**.


### Preprocessing the spots

Before running PCIseq, the ISS spots must be preprocessed and aligned with the scRNAseq dataset:

1. **Filter spots by quality**
   Spots can be filtered using various quality metrics. Please refer to the manual for details about available metrics and recommended thresholds.

2. **Coordinate conversion**

   * If ISS spot coordinates are already in **pixels**, use
     `conversion_factor = 1`.

   * If coordinates are in **micrometers**, convert them into pixel units (to match the segmentation mask) using for example
     `conversion_factor = 0.1625`.

3. **Gene intersection**
   To ensure comparability, we restrict both ISS and scRNAseq data to the set of genes present in *both* modalities. Genes absent from one dataset are not informative for integration and are excluded.

### Data flow

Here’s how the pieces fit together in the pipeline:

**Decoded ISS spots** (`spots_file`)
⬇ filter + convert + gene intersection

**Segmentation mask** (`coo_file`)
⬇ assign spots → cells

**scRNAseq reference**
⬇ restrict to shared genes

➡️ **Spatial gene-by-cell matrix** → input to **PCIseq**


### Input Parameters

**`input_dir`** *(str)*
Path to the parent directory containing the **preprocessed region folders**
(e.g., `/R1/`, `/R2/`, …). These folders are created automatically during preprocessing and contain decoded spots, segmentation masks, and downstream results.

**`region`** *(str)*
Region identifier to process (e.g., `"R1"`). The pipeline will load decoded spots and store PCIseq results inside this region folder.

**`segmentation_method`** *(str)*
A descriptive name or label for the segmentation used (`"cellpose"` or `"stardist"` are supported).

When `segmentation_file` is **not provided**, the segmentation mask is expected to follow the standard pipeline layout and will be loaded from:

```
<input_dir>/<region>/postprocessing/segmentation/{region}_{segmentation_method}_expanded.npz
```

For example:

```
/data/experiment/R1/postprocessing/segmentation/R1_cellpose_expanded.npz
```

**`segmentation_file`** *(str or Path, optional)*
Full path to a segmentation mask (`.npz` file).

If provided, this file will be used directly instead of constructing the default path. This allows segmentation masks to be stored in **custom locations or naming schemes**.

This option should be used when the segmentation **does not follow the standard pipeline layout**, for example when using **Baysor-based segmentation**, where masks may be generated and stored outside the default segmentation directory.

Example:
```
segmentation_file = Path("/custom/path/R1_baysor_mask.npz")
```

If `None`, the segmentation file will be loaded using the standard path pattern described above.

**`dense`** *(bool, default=True)*
Which decoding output to use (`dense` or sparse).
This must match the decoded spots saved during the decoding step.

**`quality_threshold`** *(float, default=0.5)*
Minimum decoding quality required for a spot to be included.


**`conversion_factor`** *(float, default=1.0)*
Factor used to convert ISS coordinates into pixel units to match the segmentation mask.

> **Note:** In this notebook, **only one region is processed at a time**.
> The variable `region` must be a **single string** (e.g., `"R1"`), not a list of regions as in earlier pipeline steps.


In [ ]:
input_dir = '/path/to/regions/'
region = 'R1'
segmentation_method = 'cellpose' # or 'stardist'
segmentation_file = None  # or Path("/custom/path/R1_baysor_mask.npz")


In [ ]:
coo, processed_spots_clean, scrnaseq_clean = PCIseq.preprocess_inputs(
    input_dir, 
    region, 
    segmentation_method,
    segmentation_file,
    scRNAseq, 
    dense=False, 
    quality_threshold=0.5, 
    conversion_factor=1.0
)


## Running PCIseq

Now we can finally run the **Probabilistic Cell Typing (PCIseq)** algorithm.  
This step integrates the ISS decoded spots with the segmentation mask and the 
reference scRNA-seq dataset to probabilistically assign each segmented cell to 
a transcriptional cell type.  

> **Note:** The current Python implementation can be quite slow, so please allow sufficient 
> time for the computation, especially when working with large datasets or many regions.


In [ ]:
PCIseq.run_pciseq(
    input_dir, 
    region,
    processed_spots_clean,   # filtered + cleaned ISS spots
    coo,                     # segmentation mask in COO format
    scrnaseq_clean,          # scRNAseq reference (overlap only)
    save_output=True
    )


# Read the PCIseq output and plot the data

PCIseq is **probabilistic** in two important ways:  
1. It calculates the probability of each cell belonging to a specific cell type (or another).  
2. It calculates the probability of each ISS spot being assigned to a given cell (or another).  

After running PCIseq, the following three output files are generated in the region’s `postprocessing/PCIseq/` folder:  

- **`most_probable.csv`**  
  A table listing each segmented cell with its `(x, y)` position, the most probable assigned cell type, and the probability of that assignment.  

- **`geneData.json`**  
  A spot-level table showing which cell each spot has been assigned to, along with the assignment probability.  

- **`cellData.json`**  
  A more detailed cell-level table that includes secondary probabilities, assignment distributions across possible cell types, and other metadata.  

We begin by reading the **`most_probable.csv`** file.


In [ ]:
PCIseq_dir = Path(input_dir) / region / "postprocessing" / "PCIseq"
pcifile = PCIseq_dir / 'most_probable.csv'
pciout = pd.read_csv(pcifile)

The `pciout['ClassName']` column will contain the primary assignment for each cell, and the `pciout['Prob']` will contain the probability of that assignment.

For each cluster, we can plot the cells, together with their color-coded PCIseq probability using the following code:

In [ ]:

# Loop over all unique assigned cell types
for cluster in pciout['ClassName'].unique():
    print(f"Plotting cluster: {cluster}")
    
    # Subset cells belonging to this cluster
    pcigene = pciout.loc[pciout['ClassName'] == cluster]
    
    # Create scatterplot (X, Y = spatial coords, colored by probability)
    plt.figure(figsize=(12, 12))
    sns.scatterplot(
        x="X", y="Y",
        hue="Prob", data=pcigene,
        palette="rainbow", s=5, edgecolor=None, alpha=0.8
    )
    
    # Titles and styling
    plt.title(f"PCIseq assignment probabilities - {cluster}", fontsize=16)
    plt.xlabel("X coordinate (pixels)")
    plt.ylabel("Y coordinate (pixels)")
    plt.legend(title="Probability", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.axis("equal")  # keep spatial aspect ratio
    plt.show()
